In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/home/s1h/Projects/ML/fraudlens/datasets/train.csv")

In [3]:
df.head()

,transaction_id,user_id,timestamp,amount,merchant_category,country,device_id,channel,hours_since_prev_txn,label
0,243747,3399,2804395,28.632151,0,11,22279,1,232259.0,0
1,88525,317,1022027,51.946133,14,14,9154,1,211366.0,0
2,214788,2095,2471000,13.033808,13,23,21154,0,33513.0,0
3,63117,2156,728428,26.546503,18,20,50757,1,4380.0,0
4,112859,4687,1304407,6.759898,2,8,12561,0,40655.0,0


In [4]:
df.isnull().sum()

transaction_id          0
user_id                 0
timestamp               0
amount                  0
merchant_category       0
country                 0
device_id               0
channel                 0
hours_since_prev_txn    0
label                   0
dtype: int64

In [5]:
print(df.select_dtypes(include='number').columns.tolist())

['transaction_id', 'user_id', 'timestamp', 'amount', 'merchant_category', 'country', 'device_id', 'channel', 'hours_since_prev_txn', 'label']


In [6]:
binary_cols = [
    col for col in df.select_dtypes(include='number').columns
    if set(df[col].dropna().unique()).issubset({0, 1})
]

print(binary_cols)

['channel', 'label']


In [7]:
df['label'].value_counts(normalize=True)

label
0    0.983583
1    0.016417
Name: proportion, dtype: float64

In [8]:
df['channel'].value_counts()

channel
0    127602
1     54523
Name: count, dtype: int64

In [9]:
df.shape

(182125, 10)

In [10]:
len(df['device_id'].value_counts())

8743

In [11]:
df[df['label'] == 1]

,transaction_id,user_id,timestamp,amount,merchant_category,country,device_id,channel,hours_since_prev_txn,label
21,198993,2121,2291532,30.810934,0,27,5728,0,112901.0,1
80,182451,2451,2102412,40.603965,14,16,24659,0,56649.0,1
147,96112,1072,1110660,25.088876,17,6,13153,0,19071.0,1
175,50511,3059,583085,17.573195,1,8,34081,0,63926.0,1
228,93721,369,1082254,46.611444,11,23,8892,0,8082.0,1
...,...,...,...,...,...,...,...,...,...,...
181938,77966,2505,899182,5.014071,8,29,17196,0,7596.0,1
181981,51142,4709,590218,39.027663,16,27,1770,1,53118.0,1
182002,92009,4148,1063153,20.073545,8,25,1284,0,94157.0,1
182007,214464,113,2467308,56.399630,7,3,28443,0,169237.0,1


In [12]:
df[df['user_id'] == 2121]['channel'].value_counts()

channel
0    25
1     9
Name: count, dtype: int64

In [13]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from xgboost import XGBClassifier

In [14]:
df = df.sort_values(["user_id", "timestamp"]).reset_index(drop=True)

# Perform any transformation for per user
user_group = df.groupby("user_id", sort=False)

## 1. Transaction

In [15]:
# Time since user's previous transaction
df['time_since_last_txn'] = user_group['timestamp'].diff()

# Average previous transaction gap
df['avg_prev_time_gap'] = (
    df.groupby('user_id')['time_since_last_txn']
    .transform(lambda x : x.shift(1)
               .rolling(10, min_periods=1)
               .mean()
    )
)

# Number of previous transactions
df["user_transaction_count"] = (
    df.groupby("user_id").cumcount()
)

# Number of transactions in the previous 10 transactions
df["transactions_prev_10"] = (
    df.groupby("user_id")["timestamp"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).count())
)

## 2. Spending behavior features

In [16]:
# Previous transaction amount
df["previous_amount"] = (
    user_group["amount"].shift(1)
)

# User's previous average amount
df["user_avg_amount_prev"] = (
    df.groupby("user_id")["amount"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# User's previous maximum amount
df["user_max_amount_prev"] = (
    df.groupby("user_id")["amount"]
    .transform(lambda x: x.shift(1).expanding().max())
)

# Amount compared with user's historical average
df["amount_vs_user_avg"] = (
    df["amount"] / (df["user_avg_amount_prev"] + 1e-6)
)

# Difference from user's previous transaction
df["amount_change"] = (
    df["amount"] - df["previous_amount"]
)

# Rolling average amount from previous 10 transactions
df["rolling_avg_amount_10"] = (
    df.groupby("user_id")["amount"]
    .transform(lambda x: x.shift(1).rolling(10, min_periods=1).mean())
)

## 3. New device

In [17]:
# Number of previous transactions using this device
df["device_previous_count"] = (
    df.groupby(["user_id", "device_id"]).cumcount()
)

# Whether this is the user's first transaction from this device
df["is_new_device"] = (
    df["device_previous_count"] == 0
).astype(int)

## 4. Country behavior features

In [18]:
# Number of previous transactions from this country
df["country_previous_count"] = (
    df.groupby(["user_id", "country"]).cumcount()
)

# Whether this is a new country for the user
df["is_new_country"] = (
    df["country_previous_count"] == 0
).astype(int)

# Previous country used by the user
df["previous_country"] = (
    df.groupby("user_id")["country"].shift(1)
)

# Whether the country changed
df["country_changed"] = (
    (df["country"] != df["previous_country"]) &
    df["previous_country"].notna()
).astype(int)

## 5. Merchant Behaviour

In [19]:
# Number of previous transactions in this merchant category
df["merchant_previous_count"] = (
    df.groupby(["user_id", "merchant_category"]).cumcount()
)

# Whether this is an unusual/new merchant category
df["is_new_merchant_category"] = (
    df["merchant_previous_count"] == 0
).astype(int)

## 6. Device and Country combination

In [20]:
# First time this device-country combination is seen
df["device_country_previous_count"] = (
    df.groupby(["user_id", "device_id", "country"]).cumcount()
)

df["new_device_country_combo"] = (
    df["device_country_previous_count"] == 0
).astype(int)

In [21]:
drop_cols = [
      "label",
      "transaction_id",
      "user_id",
      "device_id",
  ]

X = df.drop(columns=drop_cols)
y = df['label']

In [22]:
# 70-15-15

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

In [23]:
X_train.head()

,timestamp,amount,merchant_category,country,channel,hours_since_prev_txn,time_since_last_txn,avg_prev_time_gap,user_transaction_count,transactions_prev_10,...,device_previous_count,is_new_device,country_previous_count,is_new_country,previous_country,country_changed,merchant_previous_count,is_new_merchant_category,device_country_previous_count,new_device_country_combo
135690,715867,115.272987,11,10,0,83633.0,83633.0,67421.714286,8,8.0,...,8,0,7,0,10.0,0,5,0,7,0
67443,16910,15.512489,5,11,1,999.0,NaN,NaN,0,0.0,...,0,1,0,1,NaN,0,0,1,0,1
16625,2571629,22.068098,15,13,0,5727.0,5727.0,79808.300000,32,10.0,...,10,0,0,1,27.0,1,21,0,0,1
70326,2103301,15.721213,8,3,0,83190.0,83190.0,64387.600000,26,10.0,...,15,0,3,0,3.0,0,18,0,2,0
123157,1503730,13.368122,7,3,0,99652.0,366090.0,52358.300000,18,10.0,...,7,0,0,1,12.0,1,0,1,0,1


In [24]:
X_train.isnull().sum()

timestamp                           0
amount                              0
merchant_category                   0
country                             0
channel                             0
hours_since_prev_txn                0
time_since_last_txn              3470
avg_prev_time_gap                6953
user_transaction_count              0
transactions_prev_10                0
previous_amount                  3470
user_avg_amount_prev             3470
user_max_amount_prev             3470
amount_vs_user_avg               3470
amount_change                    3470
rolling_avg_amount_10            3470
device_previous_count               0
is_new_device                       0
country_previous_count              0
is_new_country                      0
previous_country                 3470
country_changed                     0
merchant_previous_count             0
is_new_merchant_category            0
device_country_previous_count       0
new_device_country_combo            0
dtype: int64

In [25]:
# Adding missing indicators before imputation

history_cols = [
    "time_since_last_txn",
    "avg_prev_time_gap",
    "previous_amount",
    "user_avg_amount_prev",
    "user_max_amount_prev",
    "amount_vs_user_avg",
    "amount_change",
    "rolling_avg_amount_10",
    "previous_country",
]

for col in history_cols:
    df[f"{col}_missing"] = df[col].isna().astype("int8")

### Imputation

In [26]:
categorical_cols = [
    "merchant_category",
    "country",
    "channel",
    "previous_country",
]

history_numeric_cols = [
    "time_since_last_txn",
    "avg_prev_time_gap",
    "previous_amount",
    "user_avg_amount_prev",
    "user_max_amount_prev",
    "amount_vs_user_avg",
    "rolling_avg_amount_10",
]

amount_change_cols = [
    "amount_change",
]

regular_numeric_cols = [
    "timestamp",
    "amount",
    "hours_since_prev_txn",
    "user_transaction_count",
    "transactions_prev_10",
    "device_previous_count",
    "is_new_device",
    "country_previous_count",
    "is_new_country",
    "country_changed",
    "merchant_previous_count",
    "is_new_merchant_category",
    "device_country_previous_count",
    "new_device_country_combo",
]

### Pipelines

In [27]:

categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-1,
        ),
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
        ),
    ),
])

history_numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=-1,
            add_indicator=True,
        ),
    ),
])

amount_change_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value=0,
            add_indicator=True,
        ),
    ),
])

regular_numeric_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True,
        ),
    ),
])

### Transformer

In [28]:
preprocessor = ColumnTransformer([
    (
          "categorical",
          categorical_pipeline,
          categorical_cols,
      ),
      (
          "history_numeric",
          history_numeric_pipeline,
          history_numeric_cols,
      ),
      (
          "amount_change",
          amount_change_pipeline,
          amount_change_cols,
      ),
      (
          "regular_numeric",
          regular_numeric_pipeline,
          regular_numeric_cols,
      ),
  ],
  remainder="drop"
)

## 1. Random forest

In [29]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200, 
            max_depth=6,
            min_samples_split=10,
            max_features="sqrt", 
            bootstrap=True, 
            max_samples=0.2,
            n_jobs=1, 
            random_state=42,
    ),
    ),
])

In [30]:
rf_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](26,)","['timestamp','amount','merchant_category',...,'is_new_merchant_category', 'device_country_previous_count','new_device_country_combo']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,26
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('history_numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolum

In [31]:
import sys
import pathlib

project_root = pathlib.Path.cwd().parent
sys.path.append(str(project_root))

from src.evaluate_model import evaluate_model
from src.threshold import calculate_threshold

In [32]:
evaluate_model(
    rf_pipeline,
    X_val,
    y_val
)

Accuracy: 0.983564552143197
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5451102934777113
PR-AUC: 0.028514115580357678

Confusion Matrix
[[26870     0]
 [  449     0]]


## XGBoost

In [33]:
spw = (y_train == 0).sum() / (y_train == 1).sum()

In [34]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            min_child_weight=1,
            gamma=0,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0,
            reg_lambda=1,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",      # fast on CPU
            n_jobs=-1,
            random_state=42,
            scale_pos_weight=spw
            # early_stopping_rounds=20,
        ),
    ),
])

In [35]:
xgb_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](26,)","['timestamp','amount','merchant_category',...,'is_new_merchant_category', 'device_country_previous_count','new_device_country_combo']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,26
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('history_numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolum

In [36]:
evaluate_model(
    xgb_pipeline,
    X_val,
    y_val
)

Accuracy: 0.8832314506387496
Precision: 0.023635731664928744
Recall: 0.1514476614699332
F1: 0.040889957907396274
ROC-AUC: 0.5046710922755194
PR-AUC: 0.017752835659315513

Confusion Matrix
[[24061  2809]
 [  381    68]]


## Tuning threshold value

In [37]:
best_threshold = calculate_threshold(rf_pipeline, X_val, y_val)

In [38]:
print(best_threshold)

0.021239329058322532


In [ ]:
evaluate_model(
    xgb_pipeline,
    X_val,
    y_val,
    threshold=best_threshold
)

Accuracy: 0.016435447856802957
Precision: 0.016435447856802957
Recall: 1.0
F1: 0.03233938346297897
ROC-AUC: 0.545839615471009
PR-AUC: 0.022848645720698897

Confusion Matrix
[[    0 26870]
 [    0   449]]


In [40]:
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

In [41]:
for split_name, X_split, y_split in [
    ("train", X_train, y_train),
    ("validation", X_val, y_val),
]:
    probabilities = rf_pipeline.predict_proba(X_split)[:, 1]

    print(split_name)
    print("AUC:", roc_auc_score(y_split, probabilities))
    print("PR-AUC:", average_precision_score(y_split, probabilities))
    print("Max probability:", probabilities.max())
    print("Quantiles:", np.quantile(probabilities, [0.5, 0.9, 0.95, 0.99, 1.0]))

train
AUC: 0.7535748743753288
PR-AUC: 0.07411668550410937
Max probability: 0.1382621510409183
Quantiles: [0.01583671 0.01846412 0.02077618 0.02746668 0.13826215]
validation
AUC: 0.5451102934777113
PR-AUC: 0.028514115580357678
Max probability: 0.08359648135771475
Quantiles: [0.01583519 0.01845402 0.02070455 0.02760522 0.08359648]


In [42]:
probabilities = rf_pipeline.predict_proba(X_val)[:, 1]

print("Non-fraud scores:")
print(np.quantile(probabilities[y_val == 0], [0.5, 0.9, 0.99, 1.0]))

print("Fraud scores:")
print(np.quantile(probabilities[y_val == 1], [0.5, 0.5, 0.9, 0.99, 1.0]))

Non-fraud scores:
[0.01583372 0.01840785 0.02734217 0.08359648]
Fraud scores:
[0.01595484 0.01595484 0.02161739 0.04016459 0.07962702]


In [43]:
from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    rf_pipeline,
    X_train,
    y_train,
    cv=5,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

print("OOF ROC-AUC:", roc_auc_score(y_train, oof_probabilities))
print("OOF PR-AUC:", average_precision_score(y_train, oof_probabilities))

OOF ROC-AUC: 0.5074716543145447
OOF PR-AUC: 0.018215332579387233


## XGBoost randomized search

In [45]:
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from scipy.stats import randint, loguniform, uniform

# Clone to avoid refitting the preprocessor shared by the existing models.
xgb_search_pipeline = clone(xgb_pipeline).set_params(
    model__n_jobs=2,
    model__early_stopping_rounds=None,
)
positive_weight = float((y_train == 0).sum() / (y_train == 1).sum())

xgb_param_distributions = {
    'model__n_estimators': randint(100, 501),
    'model__learning_rate': loguniform(0.01, 0.15),
    'model__max_depth': randint(2, 6),
    'model__min_child_weight': loguniform(5, 100),
    'model__gamma': uniform(0, 5),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.5, 0.5),
    'model__reg_alpha': loguniform(1e-3, 10),
    'model__reg_lambda': loguniform(1, 100),
    'model__scale_pos_weight': [1.0, positive_weight ** 0.5, positive_weight],
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_search_pipeline,
    param_distributions=xgb_param_distributions,
    n_iter=20,
    scoring='average_precision',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    refit=True,
    return_train_score=True,
    n_jobs=2,
    pre_dispatch=2,
    random_state=42,
    verbose=2,
    error_score='raise',
)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_
print('Best parameters:', xgb_search.best_params_)
print('Best CV average precision:', xgb_search.best_score_)
print('Training fraud prevalence:', y_train.mean())

xgb_search_results = pd.DataFrame(xgb_search.cv_results_)
display(xgb_search_results.sort_values('rank_test_score')[[
    'rank_test_score', 'mean_train_score', 'mean_test_score',
    'std_test_score', 'mean_fit_time', 'params',
]].head(10))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[CV] END model__colsample_bytree=0.6872700594236812, model__gamma=4.75357153204958, model__learning_rate=0.07259248719561363, model__max_depth=2, model__min_child_weight=29.887525711801285, model__n_estimators=221, model__reg_alpha=0.004207053950287938, model__reg_lambda=1.3066739238053278, model__scale_pos_weight=1.0, model__subsample=0.8404460046972835; total time=   1.8s
[CV] END model__colsample_bytree=0.6872700594236812, model__gamma=4.75357153204958, model__learning_rate=0.07259248719561363, model__max_depth=2, model__min_child_weight=29.887525711801285, model__n_estimators=221, model__reg_alpha=0.004207053950287938, model__reg_lambda=1.3066739238053278, model__scale_pos_weight=1.0, model__subsample=0.8404460046972835; total time=   2.2s
[CV] END model__colsample_bytree=0.6872700594236812, model__gamma=4.75357153204958, model__learning_rate=0.07259248719561363, model__max_depth=2, model__min_child_weight=29.887525711801

,rank_test_score,mean_train_score,mean_test_score,std_test_score,mean_fit_time,params
4,1,0.093281,0.020505,0.001529,2.714591,{'model__colsample_bytree': 0.9041986740582306...
8,2,0.033713,0.020318,0.002517,1.001419,{'model__colsample_bytree': 0.8534286719238086...
16,3,0.028358,0.020238,0.001883,1.630965,{'model__colsample_bytree': 0.6333905071376424...
3,4,0.024858,0.020179,0.002027,0.795030,{'model__colsample_bytree': 0.6912309956335814...
11,5,0.047215,0.020143,0.002079,2.233516,{'model__colsample_bytree': 0.8182052056318903...
14,6,0.033780,0.019965,0.001839,1.841305,{'model__colsample_bytree': 0.5034760652655954...
15,7,0.025969,0.019830,0.001684,1.951861,{'model__colsample_bytree': 0.7790510010086706...
6,8,0.068142,0.019724,0.000929,2.007533,{'model__colsample_bytree': 0.6632703844029177...
19,9,0.028999,0.019589,0.001219,1.413977,{'model__colsample_bytree': 0.9047505230698577...
0,10,0.027640,0.019556,0.001479,1.673757,{'model__colsample_bytree': 0.6872700594236812...


In [81]:
evaluate_model(
    xgb_pipeline,
    X_val,
    y_val
)

Accuracy: 0.8832314506387496
Precision: 0.023635731664928744
Recall: 0.1514476614699332
F1: 0.040889957907396274
ROC-AUC: 0.5046710922755194
PR-AUC: 0.017752835659315513

Confusion Matrix
[[24061  2809]
 [  381    68]]


In [46]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

for split_name, X_split, y_split in [
    ('train', X_train, y_train),
    ('validation', X_val, y_val),
]:
    scores = best_xgb.predict_proba(X_split)[:, 1]
    print(split_name, {
        'ROC-AUC': roc_auc_score(y_split, scores),
        'average_precision': average_precision_score(y_split, scores),
        'fraud_prevalence': float(y_split.mean()),
    })

# Select a threshold specifically for the tuned XGBoost model.
xgb_val_scores = best_xgb.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, xgb_val_scores)
f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(
    precision[:-1] + recall[:-1], 1e-12
)
xgb_best_threshold = float(thresholds[np.argmax(f1)])
print('Validation-selected XGBoost threshold:', xgb_best_threshold)
# These threshold-dependent validation metrics are tuning results.
evaluate_model(best_xgb, X_val, y_val, threshold=xgb_best_threshold)

train {'ROC-AUC': 0.7966414143555967, 'average_precision': 0.07596808674711725, 'fraud_prevalence': 0.016417360201432304}
validation {'ROC-AUC': 0.545839615471009, 'average_precision': 0.022848645720698897, 'fraud_prevalence': 0.016435447856802957}
Validation-selected XGBoost threshold: 0.1573897898197174
Accuracy: 0.9603572605146601
Precision: 0.047142857142857146
Recall: 0.07349665924276169
F1: 0.057441253263707574
ROC-AUC: 0.545839615471009
PR-AUC: 0.022848645720698897

Confusion Matrix
[[26203   667]
 [  416    33]]


In [89]:
precision, recall, thresholds = precision_recall_curve(
    y_val, best_xgb.predict_proba(X_val)[:, 1]
)

target_recall = 0.55
eligible = np.flatnonzero(recall[:-1] >= target_recall)

# Highest precision among thresholds meeting the recall target.
best_index = eligible[np.argmax(precision[:-1][eligible])]
recall_threshold = float(thresholds[best_index])

evaluate_model(
    best_xgb, X_val, y_val,
    threshold=recall_threshold,
)

Accuracy: 0.5169296094293349
Precision: 0.018798127736675222
Recall: 0.5545657015590201
F1: 0.03636363636363636
ROC-AUC: 0.545839615471009
PR-AUC: 0.022848645720698897

Confusion Matrix
[[13873 12997]
 [  200   249]]


In [90]:
from pathlib import Path
import joblib

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / "xgb_recall_055.joblib"

joblib.dump(
    {
        "pipeline": best_xgb,
        "threshold": recall_threshold,
        "target_recall": target_recall,
        "feature_columns": list(best_xgb.feature_names_in_),
    },
    artifact_path,
)

print("Saved:", artifact_path.resolve())

Saved: /home/s1h/Projects/ML/fraudlens/notebooks/artifacts/xgb_recall_055.joblib


## Baseline

In [63]:
numeric_features = [
    "amount",
    "hours_since_prev_txn",
]

categorical_features = [
    "merchant_category",
    "country",
    "channel",
]

baseline_features = [
    "amount",
    "hours_since_prev_txn",
    "merchant_category",
    "country",
    "channel",
]

In [64]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",  # Exclude every engineered feature.
)

In [65]:
rf_baseline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        min_samples_leaf=50,
        max_features="sqrt",
        class_weight="balanced_subsample",
        n_jobs=2,
        random_state=42,
    )),
])

positive_count = int((y_train == 1).sum())
negative_count = int((y_train == 0).sum())

if positive_count == 0 or negative_count == 0:
    raise ValueError("Training data must contain both fraud and non-fraud.")

xgb_baseline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=1.0,
        reg_lambda=10.0,
        scale_pos_weight=negative_count / positive_count,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        n_jobs=2,
        random_state=42,
    )),
])

In [ ]:
models = {
    "Random Forest": rf_baseline,
    "XGBoost": xgb_baseline,
}

for name, model in models.items():
    model.fit(X_train[baseline_features], y_train)

    print(f"\n{name}")

    for split_name, X_split, y_split in [
        ("train", X_train, y_train),
        ("validation", X_val, y_val),
    ]:
        scores = model.predict_proba(X_split[baseline_features])[:, 1]

        print(
            f"{split_name}: "
            f"ROC-AUC={roc_auc_score(y_split, scores):.4f}, "
            f"AP={average_precision_score(y_split, scores):.4f}, "
            f"fraud prevalence={y_split.mean():.4f}"
        )

    evaluate_model(
        model,
        X_val,
        y_val
    )


Random Forest
train: ROC-AUC=0.6535, AP=0.0326, fraud prevalence=0.0164
validation: ROC-AUC=0.5128, AP=0.0184, fraud prevalence=0.0164
Accuracy: 0.7042351476994033
Precision: 0.019155639571518588
Recall: 0.33853006681514475
F1: 0.03625954198473282
ROC-AUC: 0.512819415100173
PR-AUC: 0.01835913431408178

Confusion Matrix
[[19087  7783]
 [  297   152]]

XGBoost
train: ROC-AUC=0.6610, AP=0.0319, fraud prevalence=0.0164
validation: ROC-AUC=0.5146, AP=0.0184, fraud prevalence=0.0164
Accuracy: 0.6258647827519309
Precision: 0.01805089761294141
Recall: 0.40757238307349664
F1: 0.03457069991499008
ROC-AUC: 0.5146048822052561
PR-AUC: 0.018407745432179025

Confusion Matrix
[[16915  9955]
 [  266   183]]
